# Classificador de Imagens com CNN do Zero — CIFAR-10

**Tema:** Visão Computacional
**Trabalho:** Seminários — Aplicações Inteligentes no Dia a Dia (N3)
**Disciplina:** Inteligência Artificial — Prof. Claudinei Dias (Ney)

Este notebook implementa, treina e avalia uma **Rede Neural Convolucional (CNN) construída do zero** (sem transfer learning) para classificar imagens do dataset **CIFAR-10**, composto por 60.000 imagens coloridas de 32x32 pixels distribuídas em 10 classes:

`avião, automóvel, pássaro, gato, cervo, cachorro, sapo, cavalo, navio, caminhão`

> **Como usar:** abra este notebook no [Google Colab](https://colab.research.google.com/), ative uma GPU em `Ambiente de execução > Alterar tipo de ambiente de execução > GPU` e execute as células em ordem (`Ambiente de execução > Executar tudo`).


## 1. Importação das bibliotecas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

print("TensorFlow:", tf.__version__)
print("GPU disponível:", tf.config.list_physical_devices('GPU'))


## 2. Carregamento e exploração do dataset

O CIFAR-10 já vem embutido no Keras, então basta chamar `cifar10.load_data()`. Ele é dividido automaticamente em 50.000 imagens de treino e 10.000 de teste.

In [ ]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

class_names = ['avião', 'automóvel', 'pássaro', 'gato', 'cervo',
               'cachorro', 'sapo', 'cavalo', 'navio', 'caminhão']

print(f"Formato de x_train: {x_train.shape}")
print(f"Formato de y_train: {y_train.shape}")
print(f"Formato de x_test:  {x_test.shape}")
print(f"Formato de y_test:  {y_test.shape}")


In [ ]:
# Visualizando algumas amostras do dataset
plt.figure(figsize=(10, 5))
for i in range(15):
    plt.subplot(3, 5, i + 1)
    plt.xticks([]); plt.yticks([]); plt.grid(False)
    plt.imshow(x_train[i])
    plt.xlabel(class_names[y_train[i][0]])
plt.tight_layout()
plt.show()


## 3. Pré-processamento

Normalizamos os valores dos pixels de `[0, 255]` para `[0, 1]`, o que acelera e estabiliza o treinamento da rede.

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# y_train e y_test já vêm como inteiros (0-9), compatíveis com
# sparse_categorical_crossentropy — não é necessário one-hot encoding.
y_train = y_train.flatten()
y_test = y_test.flatten()

print("Valores de pixel após normalização -> min:", x_train.min(), "max:", x_train.max())


## 4. Definição da arquitetura da CNN

Arquitetura simples, construída do zero (sem uso de modelos pré-treinados):

```
Entrada (32x32x3)
   ↓
Conv2D (32 filtros, 3x3) + ReLU
MaxPooling2D (2x2)
   ↓
Conv2D (64 filtros, 3x3) + ReLU
MaxPooling2D (2x2)
   ↓
Conv2D (64 filtros, 3x3) + ReLU
   ↓
Flatten
Dense (64) + ReLU
Dropout (0.5)
Dense (10) + Softmax
```

In [ ]:
model = models.Sequential([
    layers.Input(shape=(32, 32, 3)),

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

model.summary()


## 5. Compilação e treinamento do modelo

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

EPOCHS = 15
BATCH_SIZE = 64

history = model.fit(
    x_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(x_test, y_test)
)


## 6. Curvas de treinamento (acurácia e perda)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(history.history['accuracy'], label='Treino')
axes[0].plot(history.history['val_accuracy'], label='Validação')
axes[0].set_title('Acurácia por época')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Acurácia')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'], label='Treino')
axes[1].plot(history.history['val_loss'], label='Validação')
axes[1].set_title('Perda (loss) por época')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Perda')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 7. Avaliação no conjunto de teste

Aqui verificamos o desempenho real do modelo em dados que ele nunca viu durante o treinamento — o indicador mais importante de qualidade do classificador.

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print(f"\nAcurácia no conjunto de teste: {test_acc:.2%}")
print(f"Perda no conjunto de teste: {test_loss:.4f}")


## 8. Matriz de confusão

A matriz de confusão mostra, para cada classe real, como as predições do modelo se distribuíram — permitindo identificar quais classes o modelo mais confunde entre si (por exemplo, gato x cachorro, ou automóvel x caminhão).

In [ ]:
y_pred_probs = model.predict(x_test)
y_pred = np.argmax(y_pred_probs, axis=1)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Classe prevista')
plt.ylabel('Classe real')
plt.title('Matriz de Confusão — CIFAR-10')
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=class_names))


## 9. Demonstração: predição em novas imagens

Função utilitária que recebe o índice de uma imagem do conjunto de teste, mostra a imagem e exibe a predição do modelo junto com o grau de confiança.

In [ ]:
def prever_imagem(indice):
    imagem = x_test[indice]
    rotulo_real = class_names[y_test[indice]]

    pred = model.predict(imagem[np.newaxis, ...], verbose=0)[0]
    classe_prevista = class_names[np.argmax(pred)]
    confianca = np.max(pred)

    plt.figure(figsize=(3, 3))
    plt.imshow(imagem)
    plt.axis('off')
    cor = 'green' if classe_prevista == rotulo_real else 'red'
    plt.title(f"Real: {rotulo_real}\nPredição: {classe_prevista} ({confianca:.1%})", color=cor)
    plt.show()

# Exemplo: testando com algumas imagens aleatórias do conjunto de teste
for i in np.random.choice(len(x_test), 5, replace=False):
    prever_imagem(i)


## 10. Salvando o modelo treinado

O modelo pode ser salvo no formato nativo do Keras (`.keras`) para ser reutilizado posteriormente sem necessidade de re-treinamento — por exemplo, em uma aplicação web ou mobile via TensorFlow Lite.

In [ ]:
model.save('cnn_cifar10.keras')
print("Modelo salvo em cnn_cifar10.keras")


## 11. Conclusão

Este notebook demonstrou, na prática, o ciclo completo de um projeto de Visão Computacional com Deep Learning:

1. Carregamento e pré-processamento de um dataset de imagens;
2. Construção de uma CNN do zero, sem uso de transfer learning;
3. Treinamento e monitoramento das métricas de treino/validação;
4. Avaliação quantitativa (acurácia, matriz de confusão, relatório de classificação);
5. Predição em novas imagens.

**Possíveis melhorias futuras:**
- **Data augmentation** (rotação, flip horizontal, zoom) para reduzir overfitting e melhorar a generalização;
- **Batch Normalization** entre as camadas convolucionais para acelerar e estabilizar o treinamento;
- **Transfer learning** com arquiteturas pré-treinadas (ResNet50, MobileNetV2) para elevar a acurácia acima de 90%;
- **Ajuste de hiperparâmetros** (taxa de aprendizado, tamanho de batch, número de filtros) via busca em grade ou Keras Tuner.
